
## Q1. What is Lakeflow in Databricks?


A **Lakeflow** is `Databricks' data engineering solution` used for **data ingestion, pipeline development** and **workflow orchestration**.


**Its main components are:**


| Component | Purpose |
|---|---|
| Lakeflow Connect | Data ingestion |
| Lakeflow Declarative Pipelines | Data transformation |
| Lakeflow Jobs | Workflow orchestration |


**Example: E-Commerce Lakehouse**


```text
CSV Files
    |
    v
Lakeflow Connect / Auto Loader
    |
    v
Bronze Tables
    |
    v
Lakeflow Declarative Pipelines
    |
    v
Silver Tables
    |
    v
Gold Tables
    |
    v
Lakeflow Jobs – Orchestration
```


## Q2. What is the difference between Imperative and Declarative Pipelines?


- **Imperative:** 

    - We specify exactly how data should be processed, step by step.

- **Declarative:** 

    -    We specify what result or dataset we want, 
    -    the framework manages how to build and maintain it.



| Feature | Imperative Pipeline | Declarative Pipeline |
|---|---|---|
| Approach | Defines how to process data | Defines what result is required |
| Coding style | Step-by-step instructions | Dataset and transformation definitions |
| Execution | Developer writes execution logic | Framework manages execution |
| Dependency management | Often handled manually | Framework can manage dependencies |
| Error handling | Developer implements error-handling logic | Framework provides built-in pipeline features |
| Maintenance | Requires more manual orchestration | Reduces manual orchestration |
| Databricks example | PySpark notebooks and Spark jobs | Lakeflow Spark Declarative Pipelines |


**Key Difference:**


- **Imperative**: 

    We explicitly control read, transform and write steps.

- **Declarative**: 

    We define the dataset and transformation; the pipeline manages execution.


## Q3. What is Lakeflow Declarative Pipelines?


**Lakeflow Declarative Pipelines** is a `Databricks framework` used to build managed `data pipelines` using **streaming tables, materialized views** and **data quality expectations**.


**Example:**



```python
from pyspark import pipelines as dp

@dp.materialized_view(
    name="gold_daily_sales"
)
def gold_daily_sales():

    return spark.sql("""
        SELECT
            order_date,
            COUNT(DISTINCT order_id) AS total_orders,
            SUM(total_amount) AS total_revenue
        FROM workspace.ecommerce_silver.orders
        GROUP BY order_date
    """)
```


## Q4. What is the difference between @dp.table and @dp.materialized_view?


- **@dp.table** defines a `streaming table`, typically used for **incremental processing of incoming data**.
- **@dp.materialized_view** defines a `materialized view`, typically used for **batch transformations** and **analytical datasets**.


| Feature | @dp.table | @dp.materialized_view |
|---|---|---|
| Dataset type | Streaming table (by default) | Materialized view |
| Main purpose | Incremental data ingestion and processing | Batch-style transformations and analytics |
| Processing | Processes new data incrementally when configured as a streaming table | Framework determines when and how to refresh results |
| Data storage | Stores data in a managed Delta table | Stores materialized query results |
| Typical use | Bronze ingestion | Silver and Gold transformations |
| Updates | Processes new records as data arrives | Recomputes or incrementally refreshes when possible |
| Example | Incoming orders | Daily sales aggregation |


**Streaming Table Example:**


**Example:** 

@dp.table (Streaming Table)


Use **@dp.table** with a `streaming DataFrame` to process `incoming data incrementally`.


**Materialized View Example:**


**Example:**

@dp.materialized_view (Materialized View)


Use **@dp.materialized_view** with a `batch DataFrame` for `transformations, aggregations and analytical datasets.`


## Q5. What is a Temporary View in Lakeflow?


- A **temporary view** is an `intermediate dataset` used inside a `pipeline`. 
- It does not create a `persistent target table`.


**Example:**


```python
from pyspark import pipelines as dp

@dp.temporary_view(
    name="completed_orders"
)
def completed_orders():

    return spark.read.table(
        "workspace.ecommerce_silver.orders"
    ).filter(
        "order_status = 'COMPLETED'"
    )
```


## Q6. What are Data Quality Expectations?


- Expectations are data quality rules used to validate records inside a Lakeflow Declarative Pipeline.
- They help identify invalid records and control how the pipeline handles them.


**Example rules:**

- `order_id` must not be NULL.
- `customer_id` must not be NULL.
- `total_amount` must be greater than zero.


**Implementation:**


```python
from pyspark import pipelines as dp

@dp.table(name="silver_orders")

@dp.expect(
    "valid_order_date",
    "order_date IS NOT NULL"
)

@dp.expect_or_drop(
    "valid_order_id",
    "order_id IS NOT NULL"
)

@dp.expect_or_fail(
    "positive_amount",
    "total_amount > 0"
)

def silver_orders():

    return spark.readStream.table(
        "workspace.ecommerce_bronze.orders"
    )
```


## Q7. What is the difference between expect, expect_or_drop and expect_or_fail?


- In **Databricks Lakeflow Pipelines**, **expectations** are `data quality rules` used to `validate records` during data processing. 
- They determine what happens when a record violates a condition.


**There are three main actions:**

- **@dp.expect** – Keep invalid records and report violations.
- **@dp.expect_or_drop** – Drop invalid records.
- **@dp.expect_or_fail** – Fail the pipeline flow when invalid records are found.


**Example:**


```python
@dp.expect(
    "valid_email",
    "email IS NOT NULL"
)

@dp.expect_or_drop(
    "valid_customer_id",
    "customer_id IS NOT NULL"
)

@dp.expect_or_fail(
    "valid_age",
    "age >= 18"
)
```



## Q8. What is expect_all in Lakeflow?


A `expect_all` allows us to define `multiple data quality rules` in a **dictionary** instead of `writing individual expectation decorators`.


**Example:**


```python
@dp.table(name="silver_customers")

@dp.expect_all({
    "valid_customer_id":
        "customer_id IS NOT NULL",

    "valid_email":
        "email IS NOT NULL",

    "valid_age":
        "age >= 18"
})

def silver_customers():

    return spark.readStream.table(
        "workspace.ecommerce_bronze.customers"
    )
```


**Other variants:**

| Function | Behavior |
|---|---|
| `expect_all` | Keeps invalid records and records metrics |
| `expect_all_or_drop` | Drops invalid records |
| `expect_all_or_fail` | Fails the affected flow update |


## Q9. What is Lakeflow Jobs?


- **Lakeflow Jobs** is used to `orchestrate and schedule multiple tasks` in a workflow.
- It supports `dependencies, parallel execution, retries, parameters, conditional tasks` and `monitoring`.


**Example:**


```text
Customer Ingestion
        |
Product Ingestion
        |
Order Ingestion
        |
        v
Lakeflow Pipeline
        |
        v
Gold Validation
        |
        v
Audit Log
```



## Q10. What is the difference between a Job and a Task?


- A **Job** is the `complete workflow` used to a`utomate and orchestrate` data processing activities. 
- A **Task** is an `individual unit of work within a job` that performs a `specific operation`.


**Simply:**

- **Job** = Complete workflow
- **Task** = Individual step inside the workflow


| Feature | Job | Task |
|---|---|---|
| Meaning | Complete workflow | Individual unit of work |
| Purpose | Orchestrates multiple activities | Executes a specific activity |
| Structure | Contains one or more tasks | Belongs to a job |
| Dependencies | Defines task execution order | Can depend on other tasks |
| Execution | Runs the complete workflow | Executes its assigned operation |
| Example | E-Commerce ETL Pipeline | Bronze ingestion |


## Q11. What is a DAG in Lakeflow Jobs?


- A **DAG** stands for `Directed Acyclic Graph`. 
- It represents `task execution dependencies` and `determines` which **tasks can run in parallel**.


**Example:**


```text
        Start
          |
    +-----+-----+
    |     |     |
    v     v     v
 Customer Product Orders
    |     |     |
    +-----+-----+
          |
          v
    Silver Pipeline
          |
          v
     Gold Tables
```


Customer, Product and Order ingestion can run in parallel if they are independent.


## Q12. What happens if an upstream task fails?


- If an upstream task fails, its `dependent tasks generally do not execute` when they require successful completion.
- We can configure retries for temporary failures and separate audit or notification tasks to handle failure scenarios.


**Example:**


```text
Ingest Orders
      |
      v
Run Pipeline
      |
      v
Validate Gold
```


If `Ingest Orders` fails, the pipeline and Gold validation should not proceed.


## Q13. What is a Retry Policy in Lakeflow Jobs?


- A **retry policy** refers to `automatically reruns a failed task` according to` configured retry limits and intervals`.
- It is useful for temporary network, storage or infrastructure failures.


**Example:**



```text
Task: Ingest_Orders

Maximum Retries: 3
Retry Interval: 2 minutes
```


**Important:** 

- Retries do not solve bad data or application logic errors. 
- The task should also be designed to handle repeated execution safely.


## Q14. What is Idempotency in Lakeflow Jobs?


**Idempotency** means `executing the same task multiple times with the same input` should **not create unintended duplicate** or **inconsistent results**.


**Example: Delta MERGE**




```sql
MERGE INTO workspace.ecommerce_silver.customers AS target

USING workspace.ecommerce_bronze.customers AS source

ON target.customer_id = source.customer_id

WHEN MATCHED THEN
    UPDATE SET
        target.customer_name = source.customer_name,
        target.city = source.city

WHEN NOT MATCHED THEN
    INSERT (
        customer_id,
        customer_name,
        city
    )
    VALUES (
        source.customer_id,
        source.customer_name,
        source.city
    );
```


**Explanation:**

- Existing customer records are updated.
- New customers are inserted.
- Reprocessing the same source snapshot does not create duplicate customer IDs.


**Important:** 

- Deduplicate the source on `customer_id` before MERGE. 
- A MERGE operation alone does not guarantee idempotency if the source contains duplicate keys.



## Q15. What is the difference between Scheduled and File Arrival Triggers?


| Scheduled Trigger | File Arrival Trigger |
|---|---|
| Starts at a configured time | Starts when new files arrive |
| Time-based | Event-based |
| Example: Daily at 1 AM | Example: New orders CSV arrives |


**Example:**


For the E-Commerce project, if orders files arrive at unpredictable times, a file-arrival trigger is more suitable than a fixed schedule.


## Q16. What is a Table Update Trigger?


A t**able update trigger** starts a Job when `a monitored Unity Catalog table or view is updated`, subject to the `supported trigger configuration`.


**Example:**


```text
Bronze Orders Updated
          |
          v
Silver Transformation Job
          |
          v
Gold Daily Sales
```


It is useful when downstream processing should react to changes in upstream tables.


## Q17. What is the difference between Parallel and Sequential Execution?


- **Parallel execution** runs `independent tasks simultaneously`. 
- **Sequential execution** runs `tasks according to dependencies`.


**Parallel Example:**


```text
      Start
        |
   +----+----+
   |         |
   v         v
Customer   Product
Ingestion  Ingestion
   |         |
   +----+----+
        |
        v
    Pipeline
```


**Sequential Example:**


```text
Bronze
  |
  v
Silver
  |
  v
Gold
```


## Q18. How do you pass parameters to a Notebook Task?


We use **notebook widgets** to pass parameters from **Databricks Lakeflow Jobs**.


**Notebook Code:**


```python
dbutils.widgets.text(
    "run_date",
    "2026-11-18"
)

dbutils.widgets.text(
    "source_dataset",
    "orders"
)

run_date = dbutils.widgets.get("run_date")

source_dataset = dbutils.widgets.get(
    "source_dataset"
)

print("Run Date:", run_date)

print("Dataset:", source_dataset)
```


**Job Parameters:**

| Parameter | Value |
|---|---|
| run_date | 2026-11-18 |
| source_dataset | orders |


## Q19. What are Task Values in Lakeflow Jobs?


- **Task Values** in **Databricks Lakeflow Job**s are `key-value pairs` used to pass `small amounts of information` from `one task` to `another` within the` same job run`.
- They allow `downstream tasks` to access values calculated by `upstream tasks` without writing those values to a separate table or file.


### Why do we use Task Values?


**Suppose you have an E-Commerce ETL workflow:**

- Task 1 – Bronze Ingestion
- Task 2 – Silver Transformation
- Task 3 – Gold Aggregation

After Bronze ingestion, you want to pass the number of records loaded to the Silver task.


Instead of storing that count in a separate table, you can use Task Values.


## Q20. How do you handle missing input files?


- I handle missing input files using  `file-availability check` before **ingestion**. 
- If required files are missing, I skip `downstream processing` and `record the missing datasets` in an audit table.


**Example:**




```python
datasets = [
    "customers",
    "products",
    "orders"
]

landing_path = (
    "/Volumes/workspace/default/data_volume/"
    "ecommerce_lakehouse/landing"
)

missing_datasets = []

for dataset in datasets:

    folder_path = (
        f"{landing_path}/{dataset}"
    )

    try:

        files = dbutils.fs.ls(folder_path)

        expected_file = f"{dataset}.csv"

        if expected_file not in [
            file.name for file in files
        ]:

            missing_datasets.append(dataset)

    except Exception as exc:

        if (
            "FileNotFoundException" in str(exc)
            or "PATH_NOT_FOUND" in str(exc)
        ):

            missing_datasets.append(dataset)

        else:

            raise

data_available = (
    len(missing_datasets) == 0
)

dbutils.jobs.taskValues.set(
    key="data_available",
    value=data_available
)

dbutils.jobs.taskValues.set(
    key="missing_datasets",
    value=missing_datasets
)
```


**Important:** 

- This is only a file-existence check. 
- Production validation should also check file readability, schema, completeness and expected business rules.


## Q21. How do you implement conditional execution in Lakeflow Jobs?


We use an **If/else task** to execute different branches based on a condition.


**Example:**


```text
       Check Data
            |
            v
          If/Else
         /       \
      TRUE       FALSE
       |           |
       v           v
    Ingest       Audit
       |
       v
    Pipeline
```



**The condition can read a task value such as:**


```text
{{tasks.check_data.values.data_available}}
```


## Q22. How do you create an audit log using SQL?


I create an audit table to track workflow execution status, timestamps and missing datasets.


**Create Table:**


```sql
CREATE TABLE IF NOT EXISTS
workspace.ecommerce_silver.audit_log
(
    event_timestamp TIMESTAMP,
    job_name STRING,
    status STRING,
    message STRING
)
USING DELTA;
```


**Insert Audit Record:**


```sql
INSERT INTO workspace.ecommerce_silver.audit_log

VALUES (
    current_timestamp(),
    'ECommerce_ETL',
    'SUCCESS',
    'Gold pipeline completed successfully'
);
```


**View Audit Records:**


```sql
SELECT *
FROM workspace.ecommerce_silver.audit_log
ORDER BY event_timestamp DESC;
```


## Q23. How do you monitor a Lakeflow Job?


- I monitor a Lakeflow Job through the `Jobs interface` and inspect `task status, execution duration, output, logs` and `error messages`.
- I also monitor **pipeline** event `logs` and `data quality metrics` to **identify failed transformations**.


**Example SQL – Check Pipeline Output Table:**


```sql
SELECT
    order_date,
    COUNT(*) AS total_orders,
    SUM(total_amount) AS total_revenue
FROM workspace.ecommerce_gold.daily_sales
GROUP BY order_date
ORDER BY order_date DESC;
```


## Q24. What is a Repair Run?


- A **Repair Run** is a `Databricks Lakeflow Jobs feature` that allows us to `rerun failed or unsuccessful tasks` in an `existing job` run without rerunning all previously successful tasks.
- It helps recover `failed workflows`, saves` execution time` and avoids `unnecessary processing`.


**Example:**


```text
Customer Ingestion  SUCCESS
Product Ingestion   SUCCESS
Order Ingestion     FAILED
Pipeline            NOT RUN
Gold Validation     NOT RUN
```



After fixing the order ingestion issue, repair the run so the failed task and required downstream tasks can execute.


## Q25. What is the difference between DAG and Data Lineage?


**DAG (Directed Acyclic Graph):**

- A **DAG** is a `graphical representation of tasks` and their `dependencies`. 
- It defines the order in which tasks should execute without creating circular dependencies.


**Data Lineage:**

- **Data Lineage** is used to tracks the `origin, movement and transformations` of data across different tables and systems.
- It helps us understand where data comes from, how it is transformed and where it is used.


| Feature | DAG (Directed Acyclic Graph) | Data Lineage |
|---|---|---|
| Full form | Directed Acyclic Graph | Data Lineage |
| Purpose | Represents task dependencies | Tracks data movement and transformations |
| Focus | Workflow execution | Data flow and history |
| Shows | Which task runs before another | Where data comes from and where it goes |
| Direction | Directed connections without cycles | Source-to-destination relationships |
| Main use | Workflow orchestration | Data governance and impact analysis |
| Databricks example | Lakeflow Jobs | Unity Catalog Lineage |


**Example:**


**DAG:**


```text

Ingest Orders
     |
     v
Run Pipeline
     |
     v
Validate Gold
```


**Data Lineage:**


```text

Bronze Orders
     |
     v
Silver Orders
     |
     v
Gold Daily Sales
```



## Q26. What is the difference between Lakeflow Connect and Auto Loader?


- **Lakeflow Connect** provides managed ingestion connectors for supported external data sources. 
- **Auto Loader** incrementally discovers and processes files from supported cloud storage locations.


| Lakeflow Connect | Auto Loader |
|---|---|
| Connects supported external sources | Processes incoming files |
| Managed ingestion connectors | File ingestion mechanism |
| Example: Supported SaaS source | Example: CSV files in a Volume |


**Auto Loader Example:**


```python
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .load(
        "/Volumes/workspace/default/data_volume/ecommerce_lakehouse/landing/orders/"
    )
)
```



## Q27. How do you prevent duplicate records during repeated ingestion?



I use source **deduplication, business keys, incremental processing** and **Delta MERGE**.


**Example:**


```python
from pyspark.sql import functions as F

source_df = spark.read.table(
    "workspace.ecommerce_bronze.orders"
)

deduplicated_df = source_df.dropDuplicates(
    ["order_id"]
)

deduplicated_df.createOrReplaceTempView(
    "source_orders"
)
```


**SQL MERGE:**


```sql
MERGE INTO workspace.ecommerce_silver.orders AS target

USING source_orders AS source

ON target.order_id = source.order_id

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *;
```


## Q28. How do you design a reliable E-Commerce Lakeflow Job?


I divide the workflow into ingestion, transformation, validation and audit stages.


**Example DAG:**


```text
              Check Files
                  |
                  v
               If/Else
              /       \
           TRUE       FALSE
            |           |
     +------+------+   Audit
     |      |      |
     v      v      v
 Customer Product Orders
     |      |      |
     +------+------+ 
            |
            v
     Lakeflow Pipeline
            |
            v
       Gold Validation
            |
            v
        Success Audit
```


I also configure retries for transient failures, task timeouts, notifications and idempotent ingestion logic.


## Q29. What happens if the Job runs twice at the same time?


- Lakeflow Jobs supports configuring the maximum number of concurrent runs. 
- If the limit is reached, additional runs can be queued when queueing is enabled or handled according to the configured concurrency policy.
- For data consistency, I also design the ingestion process to handle overlapping runs safely using batch identifiers, watermarks, deduplication and Delta MERGE.


## Q30. Explain your E-Commerce Lakehouse orchestration architecture in an interview.


- I use Lakeflow Jobs to orchestrate the complete E-Commerce Lakehouse workflow.
- First, I validate the availability of the seven source datasets: customers, products, orders, order items, payments, shipments and inventory snapshots.
- Independent ingestion tasks run in parallel and load data into Bronze Delta tables.
- After all required ingestion tasks complete successfully, the Lakeflow Declarative Pipeline processes the Bronze data into Silver and Gold tables.
- I use data quality expectations to validate important business rules, Delta MERGE for incremental processing and audit tables to track execution status.
- Finally, I validate the Gold tables before making the data available for SQL dashboards and reporting.


## 13. Final Interview Revision:


| Topic | Important Point |
|---|---|
| Lakeflow | Databricks data engineering solution |
| Lakeflow Connect | Data ingestion |
| Lakeflow Declarative Pipelines | Managed data transformations |
| Lakeflow Jobs | Workflow orchestration |
| Imperative | Explicitly write execution steps |
| Declarative | Define datasets and transformations |
| `@dp.table` | Streaming table |
| `@dp.materialized_view` | Materialized view |
| `@dp.temporary_view` | Pipeline-scoped temporary view |
| `expect` | Keep invalid rows and record metrics |
| `expect_or_drop` | Drop invalid rows |
| `expect_or_fail` | Fail affected flow update |
| Job | Complete workflow |
| Task | Individual operation |
| DAG | Task execution dependencies |
| Trigger | Starts a Job |
| Parameter | Runtime input |
| Retry | Re-executes a failed task |
| Task Values | Pass values between tasks |
| If/else | Conditional execution |
| Idempotency | Safe repeated execution |
| Repair Run | Rerun failed or selected tasks and required downstream tasks |
| Monitoring | Inspect task output, logs and run history |
| Data Lineage | Trace upstream and downstream data dependencies |



## Important Interview Preparation Advice:

For every scenario, explain these five things:

1. **Business Requirement:** What problem are you solving?
2. **Databricks Feature:** Which Lakeflow feature would you choose?
3. **Implementation:** How would you implement it using SQL or PySpark?
4. **Failure Handling:** What happens if the implementation fails?
5. **Reason:** Why did you choose this solution instead of an alternative?

Do not memorize API names alone. Understand the pipeline execution model, data quality behavior, task dependencies, failure handling and idempotency.